# OCDP + Snowflake Iceberg Readiness POC - Environment Setup

## Success Criteria Reference
| ID | Criteria | Target |
|----|----------|--------|
| SC-01 | Performance Parity | Read/write latency within <10-15% of native tables |
| SC-02 | Feature Compatibility | 100% pass rate for GA features |
| SC-03 | Governance Integrity | Horizon policies enforced with zero leakage |
| SC-04 | Resilience (HA/DR) | Successful regional failover meeting RTO/RPO |
| SC-05 | Cost Predictability | Documented TCO comparison |

## Notebook Series
| # | Notebook | Purpose |
|---|----------|--------|
| 00 | Setup Environment | Infrastructure & baseline data |
| 01 | Iceberg V3 Basics | VARIANT, nanosecond timestamps |
| 02 | Performance Benchmarks | Native vs Iceberg comparison |
| 03 | DML Operations | INSERT/UPDATE/DELETE/MERGE, time travel |
| 04 | Streams/Tasks/DT | CDC, Dynamic Tables |
| 05 | Governance | Masking, RAP, Tags |
| 06 | HA/DR | Replication templates |
| 07 | Concurrency | Stress testing |
| 08 | Databricks Interop | CLD + Unity Catalog |
| 09 | Results Summary | POC validation |

In [ ]:
-- Set context
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;

---
## Step 1: Create POC Database and Schemas

In [ ]:
-- Create POC Database
CREATE DATABASE IF NOT EXISTS ICEBERG_POC;
USE DATABASE ICEBERG_POC;

-- Create schemas
CREATE SCHEMA IF NOT EXISTS TESTS;           -- Iceberg tables for testing
CREATE SCHEMA IF NOT EXISTS NATIVE_BASELINE; -- Native tables for comparison
CREATE SCHEMA IF NOT EXISTS NOTEBOOKS;       -- For notebook deployment

SHOW SCHEMAS IN DATABASE ICEBERG_POC;

---
## Step 2: Verify External Volume

In [ ]:
-- Check External Volume exists
SHOW EXTERNAL VOLUMES LIKE 'EXVOL';

In [ ]:
-- Describe External Volume configuration
DESCRIBE EXTERNAL VOLUME EXVOL;

---
## Step 3: Create Warehouses for Benchmarking

In [ ]:
-- Create warehouses of different sizes for performance testing
CREATE WAREHOUSE IF NOT EXISTS POC_WH_XS 
    WITH WAREHOUSE_SIZE = 'XSMALL' 
    AUTO_SUSPEND = 60 
    AUTO_RESUME = TRUE;

CREATE WAREHOUSE IF NOT EXISTS POC_WH_M 
    WITH WAREHOUSE_SIZE = 'MEDIUM' 
    AUTO_SUSPEND = 60 
    AUTO_RESUME = TRUE;

CREATE WAREHOUSE IF NOT EXISTS POC_WH_L 
    WITH WAREHOUSE_SIZE = 'LARGE' 
    AUTO_SUSPEND = 60 
    AUTO_RESUME = TRUE;

SHOW WAREHOUSES LIKE 'POC_WH%';

---
## Step 4: Create Baseline Datasets
Create identical datasets in both Native and Iceberg formats for comparison testing.

In [ ]:
-- Create Native baseline table
CREATE OR REPLACE TABLE ICEBERG_POC.NATIVE_BASELINE.EVENTS (
    event_id INT AUTOINCREMENT,
    customer_id INT,
    event_type STRING,
    event_timestamp TIMESTAMP_NTZ(9),
    event_data VARIANT,
    region STRING,
    amount DECIMAL(18,2)
);

In [ ]:
-- Create Iceberg v3 equivalent table
CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.TESTS.EVENTS (
    event_id INT,
    customer_id INT,
    event_type STRING,
    event_timestamp TIMESTAMP_NTZ(9),
    event_data VARIANT,
    region STRING,
    amount DECIMAL(18,2)
)
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'EXVOL'
ICEBERG_VERSION = 3;

In [ ]:
-- Generate 1M rows of sample data for Native table
INSERT INTO ICEBERG_POC.NATIVE_BASELINE.EVENTS 
    (customer_id, event_type, event_timestamp, event_data, region, amount)
SELECT 
    UNIFORM(1, 100000, RANDOM()) AS customer_id,
    ARRAY_CONSTRUCT('login', 'purchase', 'view', 'logout', 'signup')[UNIFORM(0, 4, RANDOM())] AS event_type,
    DATEADD('second', -UNIFORM(0, 86400*365, RANDOM()), CURRENT_TIMESTAMP())::TIMESTAMP_NTZ(9) AS event_timestamp,
    OBJECT_CONSTRUCT(
        'device', ARRAY_CONSTRUCT('mobile', 'desktop', 'tablet')[UNIFORM(0, 2, RANDOM())],
        'browser', ARRAY_CONSTRUCT('Chrome', 'Safari', 'Firefox', 'Edge')[UNIFORM(0, 3, RANDOM())],
        'session_id', UUID_STRING()
    ) AS event_data,
    ARRAY_CONSTRUCT('US-EAST', 'US-WEST', 'EU', 'APAC')[UNIFORM(0, 3, RANDOM())] AS region,
    ROUND(UNIFORM(1, 10000, RANDOM()) / 100.0, 2) AS amount
FROM TABLE(GENERATOR(ROWCOUNT => 1000000));

In [ ]:
-- Copy data to Iceberg table
INSERT INTO ICEBERG_POC.TESTS.EVENTS
SELECT * FROM ICEBERG_POC.NATIVE_BASELINE.EVENTS;

In [ ]:
-- Verify row counts match
SELECT 'Native' AS table_type, COUNT(*) AS row_count FROM ICEBERG_POC.NATIVE_BASELINE.EVENTS
UNION ALL
SELECT 'Iceberg' AS table_type, COUNT(*) AS row_count FROM ICEBERG_POC.TESTS.EVENTS;

---
## Step 5: Verify Iceberg Table Metadata

In [ ]:
-- Show all Iceberg tables
SHOW ICEBERG TABLES IN SCHEMA ICEBERG_POC.TESTS;

In [ ]:
-- Get Iceberg metadata information
SELECT SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.TESTS.EVENTS') AS iceberg_info;

---
## Step 6: Sample Data Validation

In [ ]:
-- Sample from Native table
SELECT * FROM ICEBERG_POC.NATIVE_BASELINE.EVENTS LIMIT 5;

In [ ]:
-- Sample from Iceberg table
SELECT * FROM ICEBERG_POC.TESTS.EVENTS LIMIT 5;

In [ ]:
-- Data distribution by region and event_type
SELECT 
    region,
    event_type,
    COUNT(*) AS event_count,
    ROUND(AVG(amount), 2) AS avg_amount
FROM ICEBERG_POC.TESTS.EVENTS
GROUP BY 1, 2
ORDER BY 1, 2;

---
## Step 7: Databricks Integration Status

In [ ]:
-- Check catalog integration for Databricks
SHOW CATALOG INTEGRATIONS LIKE 'gf_interop%';

In [ ]:
-- Check CLD database
SHOW DATABASES LIKE 'gf_interop_cld';

In [ ]:
-- List CLD tables (from Databricks Unity Catalog)
SHOW ICEBERG TABLES IN DATABASE gf_interop_cld;

---
## Setup Complete

### Infrastructure Created
- **Database**: ICEBERG_POC
- **Schemas**: TESTS, NATIVE_BASELINE, NOTEBOOKS
- **Warehouses**: POC_WH_XS, POC_WH_M, POC_WH_L
- **Baseline Data**: 1M rows in both Native and Iceberg formats

### External Dependencies
- **External Volume**: EXVOL (Azure Blob Storage)
- **CLD Database**: gf_interop_cld (Databricks Unity Catalog)

### Next Steps
Run the remaining notebooks in order:
1. `01_Iceberg_V3_Basics.ipynb` - Test VARIANT and timestamps
2. `02_Performance_Benchmarks.ipynb` - Compare performance
3. Continue through notebook 09...